# An Introduction to Z3

This notebook gives a short introduction to the SMT solver `Z3`.  Most of the examples are taken from the [Z3 Python tutorial](https://ericpony.github.io/z3py-tutorials/guide-examples.htm).  We use the `.NET` interface of `Z3` together with the small layer defined in the file `z3.fsx`, which makes it possible to write constraints in a similar way as in Python.

In [1]:
// load the formatting setup and the style sheet, install Z3 if necessary
#load "../setup.fsx"
installZ3 ()
HTML(System.IO.File.ReadAllText "../style.css")

The below script needs to be able to find the current output cell; this is an easy method to get it.

In [2]:
// load Z3
#load "../z3.fsx"
open Microsoft.Z3

## Basics

The functions `Int`, `Real`, and `Bool` declare variables.  The function `solve` takes a list of constraints and prints a solution.  Equations and comparisons are written with a leading dot: `.=`, `.<>`, `.<`, `.<=`, `.>`, `.>=`.

In [3]:
let x = Int "x"
let y = Int "y"
solve [ x .> 2; y .< 10; x + 2*y .= 7 ]

[y = 0, x = 7]

Every Z3 expression has a type.  Integer variables have the type `IntExpr`, while constraints have the type `BoolExpr`.

In [4]:
x.GetType().Name, (x .> 2).GetType().Name

("IntExpr", "BoolExpr")

The function `simplify` simplifies an expression.

In [5]:
simplify (x + y + 2*x + 3)

3 + 3 * x + y

In [6]:
simplify (x .< y + x + 2)

¬(y ≤ -2)

In [7]:
simplify (And [ x + 1 .>= 3; x .** 2 + x .** 2 + y .** 2 + 2 .>= 5 ])

x ≥ 2 ∧ 2 * x ** 2 + y ** 2 ≥ 3

Expressions are trees.  The property `Args` returns the arguments of an expression, while `FuncDecl` returns the function symbol at the root of the tree.

In [8]:
let n = x + y .>= 3
printfn "num args:  %d" n.NumArgs
printfn "children:  %s" (String.concat ", " [ for a in n.Args -> z3ToString a ])
printfn "1st child: %s" (z3ToString n.Args[0])
printfn "2nd child: %s" (z3ToString n.Args[1])
printfn "operator:  %s" (n.FuncDecl.ToString())
printfn "op name:   %s" (n.FuncDecl.Name.ToString())

num args:  

2

children:  

x + y, 3

1st child: 

...

3

operator:  

(declare-fun >= (Int Int) Bool)

op name:   

>=

## Real Numbers

Z3 can also solve non-linear equations over the real numbers.  Irrational solutions are displayed as decimal numbers.  The trailing `?` indicates that the decimal representation has been truncated.

In [9]:
let xr = Real "x"
let yr = Real "y"
solve [ xr .** 2 + yr .** 2 .= 3; xr + yr .= 2 ]

[y = 0.2928932188?, x = 1.7071067811?]

In [10]:
solve [ xr .** 2 + yr .** 2 .= 3; xr .** 3 .= 2 ]

[y = -1.1885280594?, x = 1.2599210498?]

The mutable variable `precision`, which is defined in `z3.fsx`, specifies the number of decimal places that are displayed.

In [11]:
precision <- 30
printfn "Solving, and displaying result with 30 decimal places"
solve [ xr .** 2 + yr .** 2 .= 3; xr .** 3 .= 2 ]

Solving, and displaying result with 30 decimal places

[y = -1.188528059421316533710369365015?, x = 1.259921049894873164767210607278?]

In F#, `1.0/3.0` is a floating point number.  In Z3, rational numbers are represented exactly.  `Q(1, 3)` creates the rational number $\frac{1}{3}$.

In [12]:
printfn "%A" (1.0 / 3.0)
printfn "%s" (z3ToString (RealVal 1.0 / 3))
printfn "%s" (z3ToString (Q(1, 3)))

0.3333333333

1 / 3

1/3

In [13]:
printfn "%s" (z3ToString (xr + Q(1, 3)))
printfn "%s" (z3ToString (xr + 0.25))

x + 1/3

x + 1/4

By default, rational numbers are displayed as fractions.  If the mutable variable `rationalToDecimal` is set to `true`, they are displayed as decimal numbers.

In [14]:
precision <- 10
solve [ 3 * xr .= 1 ]

[x = 1/3]

In [15]:
rationalToDecimal <- true
solve [ 3 * xr .= 1 ]

[x = 0.3333333333?]

In [16]:
precision <- 60
solve [ 3 * xr .= 1 ]

[x = 0.333333333333333333333333333333333333333333333333333333333333?]

In [17]:
rationalToDecimal <- false
precision <- 10

If the constraints have no solution, `solve` prints `no solution`.

In [18]:
solve [ xr .> 4; xr .< 0 ]

no solution

## Boolean Logic

In [19]:
let p = Bool "p"
let q = Bool "q"
let r = Bool "r"
solve [ Implies(p, q); r .= Not q; Or [ Not p; r ] ]

[r = false, q = true, p = false]

In [20]:
printfn "%s" (z3ToString (And [ p; q; BoolVal true ]))
printfn "%s" (z3ToString (simplify (And [ p; q; BoolVal true ])))
printfn "%s" (z3ToString (simplify (And [ p; BoolVal false ])))

p ∧ q ∧ true

p ∧ q

false

## Solvers

The function `newSolver` creates a solver.  Constraints are added with the method `Add` and the method `Check` checks whether the constraints are satisfiable.  The method `Push` creates a new scope.  All constraints that are added after `Push` are removed by the method `Pop`.

In [21]:
let s = newSolver ()
print s
s.Add(x .> 10, y .= x + 2)
print s
printfn "Solving constraints in the solver s ..."
print (s.Check())
printfn "Create a new scope..."
s.Push()
s.Add(y .< 11)
print s
printfn "Solving updated set of constraints..."
print (s.Check())
printfn "Restoring state..."
s.Pop()
print s
printfn "Solving restored set of constraints..."
print (s.Check())

[]

[x > 10, y = x + 2]

Solving constraints in the solver s ...

sat

Create a new scope...

...

unsat

Restoring state...

[x > 10, y = x + 2]

Solving restored set of constraints...

sat

Z3 can not solve every problem.  For example, it is not able to solve the equation $2^x = 3$.

In [22]:
let s2 = newSolver ()
s2.Add(2 .** xr .= 3)
s2.Check()

unknown

The property `Assertions` returns the constraints of a solver, while `Statistics` returns statistical information about the last call of `Check`.

In [23]:
let s3 = newSolver ()
s3.Add(xr .> 1, yr .> 1, Or [ xr + yr .> 3; xr - yr .< 2 ])
printfn "asserted constraints..."
for c in s3.Assertions do
    print c

asserted constraints...

x > 1

y > 1

x + y > 3 ∨ x - y < 2

In [24]:
print (s3.Check())
printfn "statistics for the last check method..."
for e in s3.Statistics.Entries do
    printfn "%s : %s" e.Key e.Value

sat

statistics for the last check method...

decisions

 : 

2

...

 : 

34.03

memory

 : 

33.64

## Models

The property `Model` of a solver returns a model of the constraints.  The method `Evaluate` returns the value of an expression in the model, while the property `ConstDecls` returns the declarations of all constants (i.e. variables) that are interpreted by the model.

In [25]:
let xs, ys, zs = Real "x", Real "y", Real "z"
let s4 = newSolver ()
s4.Add(xs .> 1, ys .> 1, xs + ys .> 3, zs - xs .< 10)
print (s4.Check())
let m = s4.Model
printfn "x = %s" (z3ToString (m.Evaluate(xs, true)))
printfn "traversing model..."
for d in m.ConstDecls do
    printfn "%s = %s" (d.Name.ToString()) (z3ToString (m.ConstInterp d))

sat

x = 

3/2

traversing model...

x

...

 = 

0

y

 = 

2

## Arithmetic

Integer and real expressions can be mixed.  The function `ToReal` converts an integer expression into a real expression.

In [26]:
let xm = Real "x"
let ym = Int "y"
let a, b, c = Real "a", Real "b", Real "c"
let sm, rm = Int "s", Int "r"
printfn "%s" (z3ToString (xm + ym + 1 + (a + sm)))
printfn "%s" (z3ToString (ToReal ym + c))

x + ToReal(y) + 1 + (a + ToReal(s))

ToReal(y) + c

The method `Simplify` can be given parameters that control the simplification.  For example, the parameter `som` puts an expression into *sum-of-monomials* form, while the parameter `mul_to_power` uses the power operator.

In [27]:
let t1 = ((xr + yr) .** 3).Simplify(ctx.MkParams().Add("som", true))
printfn "%s" (z3ToString t1)
let t2 = t1.Simplify(ctx.MkParams().Add("mul_to_power", true))
printfn "%s" (z3ToString t2)

x * x * x + 3 * (x * x * y) + 3 * (x * y * y) + y * y * y

x ** 3 + 3 * (x ** 2 * y) + 3 * (x * y ** 2) + y ** 3

The method `ctx.SimplifyHelp()` returns a description of all parameters of `Simplify`.  As this description is quite long, we only print its beginning.

In [28]:
printfn "%s" (ctx.SimplifyHelp().Substring(0, 1000))

algebraic_number_evaluator (bool) simplify/evaluate expressions containing (algebraic) irrational numbers. (default: true)
arith_ineq_lhs (bool) rewrite inequalities so that right-hand-side is a constant. (default: false)
arith_lhs (bool) all monomials are moved to the left-hand-side, and the right-hand-side is just a constant. (default: false)
bit2bool (bool) try to convert bit-vector terms of size 1 into Boolean terms (default: true)
blast_distinct (bool) expand a distinct predicate into a quadratic number of disequalities (default: false)
blast_distinct_threshold (unsigned int) when blast_distinct is true, only distinct expressions with less than this number of arguments are blasted (default: 4294967295)
blast_eq_value (bool) blast (some) Bit-vector equalities into bits (default: false)
blast_select_store (bool) eagerly replace all (select (store ..) ..) term by an if-then-else term (default: false)
bv_extract_prop (bool) attempt to partially propagate extraction inwards (default: f

Z3 supports arbitrarily large numbers.  As these numbers do not fit into an F# integer, we create them from strings.  The function `Sqrt` computes square roots.  The method `ToString` displays an expression in the syntax of Z3, which is based on *s-expressions*.

In [29]:
solve [ xr + ctx.MkReal("10000000000000000000000") .= yr; yr .> ctx.MkReal("20000000000000000") ]
printfn "%s" (z3ToString (Sqrt (RealVal 2.0) + Sqrt (RealVal 3.0)))
printfn "%s" (z3ToString (simplify (Sqrt (RealVal 2.0) + Sqrt (RealVal 3.0))))
printfn "%s" ((simplify (Sqrt (RealVal 2.0) + Sqrt (RealVal 3.0))).ToString())
printfn "%s" ((xr + Sqrt yr * 2).ToString())

[x = -9999979999999999999999, y = 20000000000000001]

2 ** (1/2) + 3 ** (1/2)

3.1462643699?

(root-obj (+ (^ x 4) (* (- 10) (^ x 2)) 1) 4)

(+ x (* (^ y (/ 1.0 2.0)) (to_real 2)))

## Bit Vectors

Bit vectors are fixed size integers.  The function `BitVec name size` declares a bit vector variable with the given number of bits.  The `.NET` interface of Z3 does not provide operators for bit vectors.  Therefore, we have to use the methods of the context `ctx`, e.g. `ctx.MkBVAdd` for addition.

In [30]:
let xb = BitVec "x" 16
let yb = BitVec "y" 16
let xPlus2 = ctx.MkBVAdd(xb, BitVecVal 2L 16)
printfn "%s" (z3ToString xPlus2)
// Internal representation
printfn "%s" (xPlus2.ToString())

x + 2

(bvadd x #x0002)

For bit vectors of length 16, $-1$ is equal to $65535$.

In [31]:
printfn "%s" (z3ToString (simplify (ctx.MkBVSub(ctx.MkBVAdd(xb, yb), BitVecVal 1L 16))))
printfn "%s" (z3ToString (simplify (BitVecVal -1L 16 .= BitVecVal 65535L 16)))
// -1 is not equal to 65535 for 32-bit integers
printfn "%s" (z3ToString (simplify (BitVecVal -1L 32 .= BitVecVal 65535L 32)))

65535 + x + y

true

false

The comparison operators for bit vectors interpret bit vectors either as signed or as unsigned numbers.  `ctx.MkBVSGT` is the signed comparison $>$, `ctx.MkBVSLT` is the signed comparison $<$.

In [32]:
let x32 = BitVec "x" 32
let y32 = BitVec "y" 32
let c32 (n: int64) = BitVecVal n 32
solve [ ctx.MkBVMul(x32, y32) .= c32 12345L
        ctx.MkBVSGT(x32, c32 1L); ctx.MkBVSGT(y32, c32 1L)
        ctx.MkBVSLT(x32, c32 12345L); ctx.MkBVSLT(y32, c32 12345L) ]

[x = 2469, y = 5]

The solution is correct, since the multiplication of bit vectors is computed modulo $2^{32}$.

In [33]:
pown 2L 31 - 1L

2147483647L

## Uninterpreted Functions

The function `Function name domain range` declares an uninterpreted function.  It is applied to arguments via `f.Apply(...)`.

In [34]:
let f = Function "f" [ intSort ] intSort
solve [ f.Apply(f.Apply(x)) .= x; f.Apply(x) .= y; x .<> y ]

[x = 0, y = 1]

## Lists of Variables

Lists of variables can be created using list comprehensions.

In [35]:
// Create two lists containing 5 integer variables
let X = [ for i in 0 .. 4 -> Int (sprintf "x%d" i) ]
let Y = [ for i in 0 .. 4 -> Int (sprintf "y%d" i) ]
X

[x0; x1; x2; x3; x4]

In [36]:
// Create a list containing X[i]+Y[i]
let X_plus_Y = [ for i in 0 .. 4 -> X[i] + Y[i] ]
X_plus_Y

[x0 + y0; x1 + y1; x2 + y2; x3 + y3; x4 + y4]

In [37]:
// Create a 3x3 "matrix" (list of lists) of integer variables
let M = [ for i in 0 .. 2 -> [ for j in 0 .. 2 -> Int (sprintf "x_%d_%d" (i+1) (j+1)) ] ]
M

[[x_1_1; x_1_2; x_1_3]; [x_2_1; x_2_2; x_2_3]; [x_3_1; x_3_2; x_3_3]]

The function `Sum` computes the sum of a list of expressions.

In [38]:
let Yr = [ for i in 0 .. 4 -> Real (sprintf "y__%d" i) ]
let P  = [ for i in 0 .. 4 -> Bool (sprintf "p__%d" i) ]
print Yr
print P
print [ for y in Yr -> y .** 2 ]
print (Sum [ for y in Yr -> y .** 2 ])

[y__0; y__1; y__2; y__3; y__4]

[p__0; p__1; p__2; p__3; p__4]

[y__0 ** 2; y__1 ** 2; y__2 ** 2; y__3 ** 2; y__4 ** 2]

y__0 ** 2 + y__1 ** 2 + y__2 ** 2 + y__3 ** 2 + y__4 ** 2

## Kinematic Equations

The following equations describe the motion of a body that accelerates uniformly:
- `d` is the distance travelled,
- `a` is the acceleration,
- `t` is the time,
- `v_i` is the initial velocity, and
- `v_f` is the final velocity.

In [39]:
let d, acc, t = Real "d", Real "a", Real "t"
let v_i, v_f = Real "v__i", Real "v__f"

In [40]:
let equations = [ d .= v_i * t + (acc * t .** 2) / 2
                  v_f .= v_i + acc * t ]
printfn "Kinematic equations:"
print equations

Kinematic equations:

[d = v__i * t + a * t ** 2 / 2; v__f = v__i + a * t]

In [41]:
// Given v_i, v_f and a, find d
let problem = [ v_i .= 30; v_f .= 0; acc .= -8 ]
printfn "Problem:"
print problem

Problem:

[v__i = 30; v__f = 0; a = -8]

In [42]:
printfn "Solution:"
solve (equations @ problem)

Solution:

[a = -8, v__i = 30, d = 225/4, v__f = 0, t = 15/4]

In [43]:
// Given v_i, t and a, find d
let problem2 = [ v_i .= 0; t .= 4.10; acc .= 6 ]
solve (equations @ problem2)
// Display rationals in decimal notation
rationalToDecimal <- true
solve (equations @ problem2)

[a = 6, v__i = 0, d = 5043/100, v__f = 123/5, t = 41/10]

[a = 6, v__i = 0, d = 50.43, v__f = 24.6, t = 4.1]